In [ ]:
 [markdown]
# # Task 2: Predictive Modeling Using Machine Learning
# **Goal:** Build a model to predict outcomes from given data.
#
# **Dataset:** Breast Cancer Wisconsin (built into scikit-learn) - predict whether a tumor is
# malignant or benign from 30 measurements. *(If your mentor gave you a dataset, replace the
# loading cell with `df = pd.read_csv("your_file.csv")` and set the target column.)*
#
# **Models:** Logistic Regression, Decision Tree, Random Forest
# **Evaluation:** Accuracy, Precision, Recall, F1, Confusion Matrix, ROC Curve

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc,
                             classification_report)

sns.set_theme(style="whitegrid")

In [ ]:
 [markdown]
# ## 1. Load and explore the data

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
df["target"] = df["target"].map({0: "malignant", 1: "benign"})  # readable labels
print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print(df["target"].value_counts())
df.head()

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x="target", data=df, palette="Set2", hue="target", legend=False)
plt.title("Class Distribution")
plt.tight_layout()
plt.savefig("01_class_distribution.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
corr = data.frame.drop(columns="target").corr()
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("02_correlation_heatmap.png", dpi=150)
plt.show()

In [ ]:
 [markdown]
# ## 2. Train / test split

In [ ]:
X = data.frame.drop(columns="target")
y = data.frame["target"]          # 1 = benign, 0 = malignant
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)

In [ ]:
 [markdown]
# ## 3. Train the models

In [ ]:
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

results, probs, preds = [], {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    preds[name], probs[name] = pred, prob
    cv = cross_val_score(model, X, y, cv=5, scoring="accuracy").mean()
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC AUC": auc(*roc_curve(y_test, prob)[:2]),
        "5-fold CV Acc": cv,
    })

results_df = pd.DataFrame(results).set_index("Model").round(4)
results_df.to_csv("model_comparison.csv")
results_df

In [ ]:
 [markdown]
# ## 4. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, pred) in zip(axes, preds.items()):
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=["malignant", "benign"]).plot(
        ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.savefig("03_confusion_matrices.png", dpi=150)
plt.show()

In [ ]:
 [markdown]
# ## 5. ROC curves

In [ ]:
plt.figure(figsize=(6, 5))
for name, prob in probs.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc(fpr, tpr):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("04_roc_curves.png", dpi=150)
plt.show()

In [ ]:
 [markdown]
# ## 6. Feature importance (Random Forest)

In [ ]:
rf = models["Random Forest"]
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values().tail(10)
plt.figure(figsize=(7, 5))
imp.plot(kind="barh", color="teal")
plt.title("Top 10 Feature Importances (Random Forest)")
plt.tight_layout()
plt.savefig("05_feature_importance.png", dpi=150)
plt.show()

In [ ]:
 [markdown]
# ## 7. Best model report

In [ ]:
best = results_df["Accuracy"].idxmax()
print("Best model:", best)
print(classification_report(y_test, preds[best], target_names=["malignant", "benign"]))

In [ ]:
 [markdown]
# ## 8. Conclusion
# - All three supervised models were trained on 80% of the data and tested on the unseen 20%.
# - Models were compared with accuracy, precision, recall, F1, ROC AUC and 5-fold cross-validation.
# - Confusion matrices and ROC curves show how many tumors each model classified correctly or wrongly.
# - Recall matters most here: missing a malignant tumor is more costly than a false alarm.
# - Future work: hyperparameter tuning (GridSearchCV), more models (SVM, XGBoost).